# IMPORTANDO AS COISAS NO PASSO 1 + CONFIG GRAVAÇÃO

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
    }
  recorder.stop()
})
"""



def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = "request_audio.wav"
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Ouvindo...\n')
record_file = record()
display(Audio(record_file, autoplay=True))


Ouvindo...



<IPython.core.display.Javascript object>

# PASSO 2 - WHISPER CONVERTE EM TEXTO

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import whisper

model = whisper.load_model('small')
result = model.transcribe(record_file, fp16=False, language='en')
transcription = result['text']
print(transcription)

 What is PlayStation?


# PASSO 3 - GOOGLE GEMINI PARA RESPONDER

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai

genai.configure(api_key="")
model = genai.GenerativeModel("models/gemini-flash-lite-latest")

response = model.generate_content("Responde em no máximo 30 palavras: " + transcription)
print(response.text)

PlayStation é uma marca de consoles de videogame da Sony, conhecida por jogos imersivos e inovações em entretenimento interativo.


# PASSO 4 - VOZ DO GOOGLE TRADUTOR PARA EXECUTAR A RESPOSTA DO GEMINI

In [ ]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [69]:
from gtts import gTTS

gtts_object = gTTS(text=response.text, lang='pt')
response_audio = "/content/responde_audio.wav"
gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))